# IMD3002 — Trabalho Final: Análise Estatística (Parte IV)

Fecha toda a parte estatística do trabalho final. **Não re-roda experimento nenhum** —
usa os F1-scores já obtidos nas práticas (transcritos dos PDFs de resultado).

**O que este notebook produz:**
1. Friedman + Nemenyi **dentro de cada técnica base** (kNN, DT, NB, MLP)
2. Friedman + Nemenyi **entre as melhores técnicas base**
3. **Comitê vs. classificador base** (houve melhora?)
4. **Melhor técnica base vs. melhor comitê** (peça central da rúbrica)
5. CD diagrams de tudo + PDF final

Roda em ~30 segundos. Métrica: F1-score CV (10-fold). Critério: $p < 0{,}05$.

## 1. Setup

In [ ]:
!pip install -q scikit-posthocs
import numpy as np, pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
import scikit_posthocs as sp
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

BASES = ['LBP_256_6R','LBP_256_12R','LBP_256_3R','HOG_128_32x32','HOG_128_16x16',
         'HOG_256_32x32','HOG_256_32x32 PCA75','HOG_128_16x16 PCA75','HOG_128_8x8',
         'HOG_128_16x16 PCA90','HOG_128_8x8 PCA90','HOG_256_32x32 PCA90']
print('Setup OK')

Setup OK


## 2. Dados dos experimentos (F1-score CV por base)
Transcritos dos PDFs de resultado de cada prática. **Confira contra seus PDFs** se quiser.

In [ ]:
# ===================== CLASSIFICADORES BASE =====================
# Cada lista = F1 (10-fold CV) por base, na ordem de BASES.

# kNN: coluna 'Media' (média sobre k=1..10) por base, do report kNN
knn = [0.6456,0.6362,0.6342,0.6278,0.4793,0.4687,0.4286,0.4223,0.4062,0.3903,0.3778,0.3717]

# DT: Conf 03 (gini, max_depth=4 — melhor média), CV por base, do report DT
dt  = [0.6721,0.6512,0.6430,0.6559,0.6309,0.6773,0.6690,0.6916,0.5883,0.6715,0.6145,0.7039]

# NB: GaussianNB (CV) por base, do report NB
nb  = [0.6583,0.6322,0.6453,0.7290,0.7578,0.7340,0.7159,0.7339,0.7529,0.7088,0.6943,0.7002]

# MLP: Conf 08 (melhor média), CV por base, do report MLP
mlp = [0.6826,0.6924,0.6853,0.7655,0.7436,0.7545,0.7703,0.7790,0.7149,0.7082,0.5477,0.7022]

base_df = pd.DataFrame({'kNN':knn,'DT':dt,'NB':nb,'MLP':mlp}, index=BASES)

# ===================== MELHORES DE CADA COMITÊ =====================
# F1 (CV) por base, do report de Comitês (melhor config de cada estrutura)
bagging_mlp15  = [0.6639,0.6902,0.6568,0.7427,0.7361,0.7487,0.7679,0.7720,0.7053,0.6836,0.5369,0.7189]
rf_entropy100  = [0.7112,0.7117,0.6807,0.7728,0.7970,0.7804,0.7741,0.7713,0.7431,0.7869,0.6999,0.7565]
stacking_20    = [0.6746,0.6782,0.6567,0.7779,0.7722,0.7785,0.7804,0.7982,0.7358,0.7705,0.6968,0.7646]
voting_15      = [0.6779,0.6516,0.6673,0.7515,0.7633,0.7415,0.7286,0.7829,0.7140,0.7549,0.6737,0.7111]

comite_df = pd.DataFrame({'Bagging MLP(15)':bagging_mlp15,'RF Entropy(100)':rf_entropy100,
                          'Stacking (20)':stacking_20,'Voting (15)':voting_15}, index=BASES)

print('Base:'); print(base_df.mean().round(4))
print('\nComitês:'); print(comite_df.mean().round(4))

Base:
kNN    0.4907
DT     0.6558
NB     0.7052
MLP    0.7122
dtype: float64

Comitês:
Bagging MLP(15)    0.7019
RF Entropy(100)    0.7488
Stacking (20)      0.7404
Voting (15)        0.7182
dtype: float64


## 3. Função genérica de análise (Friedman + Nemenyi + CD)

In [ ]:
def analise(df, titulo, png=None):
    cols = list(df.columns)
    stat, p = friedmanchisquare(*[df[c].values for c in cols])
    ranks = df.rank(axis=1, ascending=False).mean()
    means = df.mean()
    print(f'\n===== {titulo} =====')
    print(f'Friedman: chi2={stat:.3f}, p={p:.4g}')
    nem = None
    if p < 0.05:
        print(f'  p < 0,05: HÁ diferença estatística.')
        if len(cols) >= 3:
            nem = sp.posthoc_nemenyi_friedman(df.values); nem.columns=nem.index=cols
        best = ranks.idxmin()
    else:
        print(f'  p >= 0,05: SEM diferença estatística.')
        best = means.idxmax()
    print('  Ranks médios (1=melhor):'); print('   '+ranks.sort_values().round(3).to_string().replace('\n','\n   '))
    print(f'  >> MELHOR: {best} (F1 médio={means[best]:.4f})')
    if png:
        plt.figure(figsize=(8,2.4))
        try:
            if nem is not None:
                sp.critical_difference_diagram(ranks, nem); plt.title(f'{titulo} (p={p:.3g})')
            else:
                ranks.sort_values().plot(kind='barh'); plt.xlabel('Rank médio (1=melhor)')
                plt.title(f'{titulo} — sem diferença (p={p:.3g})')
            plt.tight_layout(); plt.savefig(png, dpi=120, bbox_inches='tight')
        finally: plt.close()
    return {'titulo':titulo,'p':p,'ranks':ranks,'means':means,'nemenyi':nem,'best':best,'png':png}

## 4. Análise ENTRE as melhores técnicas base

In [ ]:
an_base = analise(base_df, 'Técnicas Base (kNN vs DT vs NB vs MLP)', '/content/cd_tecnicas_base.png')
if an_base['nemenyi'] is not None:
    print('\nNemenyi (p-values):'); print(an_base['nemenyi'].round(4))


===== Técnicas Base (kNN vs DT vs NB vs MLP) =====
Friedman: chi2=23.500, p=3.176e-05
  p < 0,05: HÁ diferença estatística.
  Ranks médios (1=melhor):
   MLP    1.500
   NB     2.000
   DT     2.583
   kNN    3.917
  >> MELHOR: MLP (F1 médio=0.7122)

Nemenyi (p-values):
        kNN      DT      NB     MLP
kNN  1.0000  0.0554  0.0016  0.0000
DT   0.0554  1.0000  0.6853  0.1679
NB   0.0016  0.6853  1.0000  0.7785
MLP  0.0000  0.1679  0.7785  1.0000


## 5. Análise ENTRE os melhores comitês

In [ ]:
an_com = analise(comite_df, 'Comitês (Bagging vs RF vs Stacking vs Voting)', '/content/cd_comites.png')
if an_com['nemenyi'] is not None:
    print('\nNemenyi (p-values):'); print(an_com['nemenyi'].round(4))


===== Comitês (Bagging vs RF vs Stacking vs Voting) =====
Friedman: chi2=17.500, p=0.0005576
  p < 0,05: HÁ diferença estatística.
  Ranks médios (1=melhor):
   RF Entropy(100)    1.500
   Stacking (20)      2.000
   Voting (15)        3.083
   Bagging MLP(15)    3.417
  >> MELHOR: RF Entropy(100) (F1 médio=0.7488)

Nemenyi (p-values):
                 Bagging MLP(15)  RF Entropy(100)  Stacking (20)  Voting (15)
Bagging MLP(15)           1.0000           0.0016         0.0362       0.9216
RF Entropy(100)           0.0016           1.0000         0.7785       0.0142
Stacking (20)             0.0362           0.7785         1.0000       0.1679
Voting (15)               0.9216           0.0142         0.1679       1.0000


## 6. Melhor técnica base VS melhor comitê
Peça central da rúbrica: houve diferença estatística entre o melhor classificador
individual e o melhor comitê? Como são 2 grupos, usa-se o teste de Wilcoxon pareado
(o Friedman exige 3+).

In [ ]:
melhor_base_nome = an_base['best']
melhor_com_nome  = an_com['best']
a = base_df[melhor_base_nome].values
b = comite_df[melhor_com_nome].values
w, pw = wilcoxon(a, b)
print(f'Melhor técnica base: {melhor_base_nome} (F1={a.mean():.4f})')
print(f'Melhor comitê:       {melhor_com_nome} (F1={b.mean():.4f})')
print(f'\nWilcoxon pareado: stat={w:.1f}, p={pw:.4g}')
if pw < 0.05:
    vencedor = melhor_com_nome if b.mean() > a.mean() else melhor_base_nome
    print(f'  p < 0,05: HÁ diferença -> {vencedor} é estatisticamente superior.')
    print(f'  >> Comitê {"MELHOROU" if b.mean()>a.mean() else "NÃO melhorou"} sobre o classificador base.')
else:
    print(f'  p >= 0,05: SEM diferença estatística entre o melhor base e o melhor comitê.')

# grafico de barras comparativo
plt.figure(figsize=(7,3))
x=np.arange(len(BASES)); plt.plot(x,a,'o-',label=f'{melhor_base_nome} (base)')
plt.plot(x,b,'s-',label=f'{melhor_com_nome} (comitê)')
plt.xticks(x,[f'B{i+1}' for i in range(len(BASES))],fontsize=7)
plt.ylabel('F1 (CV)'); plt.legend(); plt.title('Melhor base vs Melhor comitê por base')
plt.tight_layout(); plt.savefig('/content/cd_base_vs_comite.png',dpi=120,bbox_inches='tight'); plt.close()
print('Gráfico salvo.')

Melhor técnica base: MLP (F1=0.7122)
Melhor comitê:       RF Entropy(100) (F1=0.7488)

Wilcoxon pareado: stat=6.0, p=0.006836
  p < 0,05: HÁ diferença -> RF Entropy(100) é estatisticamente superior.
  >> Comitê MELHOROU sobre o classificador base.
Gráfico salvo.


## 7. Comitê melhorou sobre seu próprio classificador base?
Compara cada comitê homogêneo com o classificador base individual correspondente.

In [ ]:
print('Comparações comitê vs. classificador individual (F1 médio CV):')
print(f'  RF Entropy(100) = {comite_df["RF Entropy(100)"].mean():.4f}  vs  DT individual = {base_df["DT"].mean():.4f}')
w1,p1 = wilcoxon(comite_df['RF Entropy(100)'], base_df['DT'])
print(f'    Wilcoxon p={p1:.4g} ({"RF superior" if p1<0.05 else "sem diferença"})')
print(f'  Bagging MLP(15) = {comite_df["Bagging MLP(15)"].mean():.4f}  vs  MLP individual = {base_df["MLP"].mean():.4f}')
w2,p2 = wilcoxon(comite_df['Bagging MLP(15)'], base_df['MLP'])
print(f'    Wilcoxon p={p2:.4g} ({"dif" if p2<0.05 else "sem diferença"})')

Comparações comitê vs. classificador individual (F1 médio CV):
  RF Entropy(100) = 0.7488  vs  DT individual = 0.6558
    Wilcoxon p=0.0004883 (RF superior)
  Bagging MLP(15) = 0.7019  vs  MLP individual = 0.7122
    Wilcoxon p=0.01221 (dif)


## 8. PDF final com todas as análises

In [ ]:
!pip install -q reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 7.8 MB/s eta 0:00:00


In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY
from reportlab.lib import colors
from reportlab.lib.units import cm
import os

OUT='/content/IMD3002_Analise_Estatistica_Final.pdf'
styles=getSampleStyleSheet()
H=ParagraphStyle('H',parent=styles['Title'],fontSize=12,leading=15,alignment=TA_CENTER,fontName='Helvetica-Bold')
SEC=ParagraphStyle('S',parent=styles['Heading2'],fontSize=11,fontName='Helvetica-Bold',spaceBefore=8)
BODY=ParagraphStyle('B',parent=styles['Normal'],fontSize=10,leading=13,alignment=TA_JUSTIFY)

doc=SimpleDocTemplate(OUT,pagesize=A4,leftMargin=2*cm,rightMargin=2*cm,topMargin=2*cm,bottomMargin=2*cm)
story=[Paragraph('Análise Estatística Final — Friedman + Nemenyi',H),
       Paragraph('IMD3002 — Trabalho Final · F1-score (10-fold CV) · α = 0,05',
                 ParagraphStyle('c',parent=styles['Normal'],fontSize=9,alignment=TA_CENTER)),
       Spacer(1,0.4*cm)]

def secao(an, extra=''):
    verdict = ('há diferença estatística' if an['p']<0.05 else 'não houve diferença estatística')
    story.append(Paragraph(an['titulo'], SEC))
    story.append(Paragraph(f"Teste de Friedman: p = {an['p']:.4g} &rarr; <b>{verdict}</b>. "
                           f"Melhor modelo: <b>{an['best']}</b> (F1 médio = {an['means'][an['best']]:.4f}). {extra}", BODY))
    if an['png'] and os.path.exists(an['png']):
        story.append(Spacer(1,0.2*cm)); story.append(Image(an['png'], width=15*cm, height=4.5*cm))
    story.append(Spacer(1,0.3*cm))

secao(an_base, 'O MLP e o NB superaram o kNN com significância.')
secao(an_com, 'O Random Forest obteve o melhor rank entre os comitês.')

# base vs comite
story.append(Paragraph('Melhor Técnica Base vs. Melhor Comitê', SEC))
vd = ('há diferença' if pw<0.05 else 'sem diferença')
story.append(Paragraph(f"{melhor_base_nome} (F1={a.mean():.4f}) vs. {melhor_com_nome} (F1={b.mean():.4f}). "
                       f"Teste de Wilcoxon pareado: p = {pw:.4g} &rarr; <b>{vd}</b>. "
                       f"{'O comitê apresentou melhora estatisticamente significativa sobre o melhor classificador individual.' if pw<0.05 and b.mean()>a.mean() else ''}", BODY))
if os.path.exists('/content/cd_base_vs_comite.png'):
    story.append(Spacer(1,0.2*cm)); story.append(Image('/content/cd_base_vs_comite.png', width=14*cm, height=6*cm))

# tabela resumo
story.append(Spacer(1,0.3*cm)); story.append(Paragraph('Resumo Geral', SEC))
data=[['Categoria','Melhor modelo','F1 médio','Friedman p'],
      ['Técnica base', an_base['best'], f"{an_base['means'][an_base['best']]:.4f}", f"{an_base['p']:.2g}"],
      ['Comitê', an_com['best'], f"{an_com['means'][an_com['best']]:.4f}", f"{an_com['p']:.2g}"],
      ['Base vs Comitê', f'{melhor_com_nome}', f"{b.mean():.4f}", f"{pw:.2g} (Wilcoxon)"]]
t=Table(data, colWidths=[3.5*cm,5*cm,2.5*cm,3.5*cm])
t.setStyle(TableStyle([('GRID',(0,0),(-1,-1),0.4,colors.black),
    ('BACKGROUND',(0,0),(-1,0),colors.HexColor('#D9E1F2')),
    ('FONTNAME',(0,0),(-1,0),'Helvetica-Bold'),('FONTSIZE',(0,0),(-1,-1),9),
    ('ALIGN',(2,0),(-1,-1),'CENTER')]))
story.append(t)

doc.build(story)
print(f'PDF gerado: {OUT} ({os.path.getsize(OUT)/1024:.1f} KB)')
from google.colab import files
files.download(OUT)

PDF gerado: /content/IMD3002_Analise_Estatistica_Final.pdf (92.8 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>